## Setup

**Note:** This test expects Sample Data to be used as datasource

In [ ]:
%run <Fundraising_Config>

In [ ]:
from pyspark.sql.functions import col

## Test 1: Age Segmentation - Concrete Examples

In [ ]:
print("=== Test 1: Age Segmentation for Specific Constituent ===\n")
print("Testing: 'Susy Contreras' (BirthDate: 1982-01-05, Expected Age: ~43, Expected Segment: 35-44)\n")

test_constituent_1 = get_gold_table("dm_Constituent") \
    .filter(col("ConstituentName").contains("Susy Contreras")) \
    .select("ConstituentKey", "ConstituentName", "Age") \
    .join(
        get_gold_table("DimConstituentSegmentBridge_AgeRange"),
        "ConstituentKey",
        "left"
    ) \
    .join(
        get_gold_table("DimConstituentSegment_AgeRange"),
        "ConstituentSegmentKey",
        "left"
    ) \
    .select("ConstituentName", "Age", "ConstituentSegmentName")

display(test_constituent_1)

## Test 2: Lifetime Giving Segmentation - Top Donors

In [ ]:
print("=== Test 2: Lifetime Giving Segmentation - Sample Constituents with Transactions ===\n")

constituents_with_donations = get_gold_table("dm_Constituent") \
    .select("ConstituentKey", "ConstituentName", "LifetimeDonationAmount") \
    .filter(col("LifetimeDonationAmount").isNotNull() & (col("LifetimeDonationAmount") > 0)) \
    .join(
        get_gold_table("DimConstituentSegmentBridge_LifetimeGivingRange"),
        "ConstituentKey",
        "left"
    ) \
    .join(
        get_gold_table("DimConstituentSegment_LifetimeGivingRange"),
        "ConstituentSegmentKey",
        "left"
    ) \
    .select("ConstituentName", "LifetimeDonationAmount", "ConstituentSegmentName") \
    .orderBy("LifetimeDonationAmount", ascending=False) \
    .limit(10)

display(constituents_with_donations)

## Test 3: Gift Recurrence - New Donors

In [ ]:
print("=== Test 3: Gift Recurrence - New Donors ===\n")

new_donors = get_gold_table("dm_Constituent") \
    .filter(col("IsNewDonor") == True) \
    .select("ConstituentKey", "ConstituentName", "IsNewDonor") \
    .join(
        get_gold_table("DimConstituentSegmentBridge_GiftRecurrence"),
        "ConstituentKey",
        "left"
    ) \
    .join(
        get_gold_table("DimConstituentSegment_GiftRecurrence"),
        "ConstituentSegmentKey",
        "left"
    ) \
    .select("ConstituentName", "IsNewDonor", "ConstituentSegmentName") \
    .limit(10)

display(new_donors)

## Test 4: Gift Recurrence - Prospects

In [ ]:
print("=== Test 4: Gift Recurrence - Prospects (No Donations) ===\n")

prospects = get_gold_table("dm_Constituent") \
    .filter(col("LastDonationDateKey").isNull()) \
    .select("ConstituentKey", "ConstituentName", "LastDonationDateKey") \
    .join(
        get_gold_table("DimConstituentSegmentBridge_GiftRecurrence"),
        "ConstituentKey",
        "left"
    ) \
    .join(
        get_gold_table("DimConstituentSegment_GiftRecurrence"),
        "ConstituentSegmentKey",
        "left"
    ) \
    .select("ConstituentName", "LastDonationDateKey", "ConstituentSegmentName") \
    .limit(10)

display(prospects)

## Summary: All Segment Counts

In [ ]:
print("\n" + "="*80)
print("FINAL SUMMARY: ALL SEGMENT MATCH COUNTS")
print("="*80 + "\n")

# Age Range Segments
print("AGE RANGE SEGMENTS:")
age_segments = get_gold_table("DimConstituentSegmentBridge_AgeRange") \
    .join(
        get_gold_table("DimConstituentSegment_AgeRange"),
        "ConstituentSegmentKey",
        "inner"
    ) \
    .groupBy("ConstituentSegmentName") \
    .count() \
    .withColumnRenamed("count", "Constituent Count") \
    .orderBy("ConstituentSegmentName")

display(age_segments)

# Lifetime Giving Range Segments
print("\nLIFETIME GIVING RANGE SEGMENTS:")
giving_segments = get_gold_table("DimConstituentSegmentBridge_LifetimeGivingRange") \
    .join(
        get_gold_table("DimConstituentSegment_LifetimeGivingRange"),
        "ConstituentSegmentKey",
        "inner"
    ) \
    .groupBy("ConstituentSegmentName") \
    .count() \
    .withColumnRenamed("count", "Constituent Count") \
    .orderBy("ConstituentSegmentName")

display(giving_segments)

# Gift Recurrence Segments
print("\nGIFT RECURRENCE SEGMENTS:")
recurrence_segments = get_gold_table("DimConstituentSegmentBridge_GiftRecurrence") \
    .join(
        get_gold_table("DimConstituentSegment_GiftRecurrence"),
        "ConstituentSegmentKey",
        "inner"
    ) \
    .groupBy("ConstituentSegmentName") \
    .count() \
    .withColumnRenamed("count", "Constituent Count") \
    .orderBy("ConstituentSegmentName")

display(recurrence_segments)

## Overall Statistics

In [ ]:
print("\n" + "="*80)
print("OVERALL STATISTICS:")
print("="*80)

total_constituents = get_gold_table("dm_Constituent").count()
age_total = age_segments.selectExpr("sum(`Constituent Count`) as total").collect()[0]["total"]
giving_total = giving_segments.selectExpr("sum(`Constituent Count`) as total").collect()[0]["total"]
recurrence_total = recurrence_segments.selectExpr("sum(`Constituent Count`) as total").collect()[0]["total"]

summary_data = [
    ("Total Constituents", total_constituents, "100.0%"),
    ("Age Segments Assigned", age_total, f"{age_total/total_constituents*100:.1f}%"),
    ("Giving Segments Assigned", giving_total, f"{giving_total/total_constituents*100:.1f}%"),
    ("Recurrence Segments Assigned", recurrence_total, f"{recurrence_total/total_constituents*100:.1f}%")
]

summary_df = spark.createDataFrame(summary_data, ["Metric", "Count", "Percentage"])
display(summary_df)

print("\n✅ All validation tests complete!")